# Read labels
- This is some code to read the output of the label tool and plot the data with the associated maxima and minima.

In [ ]:
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from helpers.df_ops import prepare_df

LABELS_PATH = Path("labels.json")
DATA_DIR    = PROJECT_ROOT / "Data" / "benchmark"
TOL         = 6

with open(LABELS_PATH) as f:
    labels = json.load(f)

print(f"Loaded labels for {len(labels)} files:")
for fname, lbl in labels.items():
    print(f"  {fname}: {len(lbl['maxima'])} maxima, {len(lbl['minima'])} minima")

In [ ]:
def load_clean(fname):
    raw_df      = pd.read_csv(DATA_DIR / fname, sep=r'\s+', skip_blank_lines=True)
    raw_data_df = prepare_df(raw_df, add_prefix=False, relative=True)
    med = raw_data_df['sind'].median()
    mad = (raw_data_df['sind'] - med).abs().median()
    data_df = raw_data_df[(raw_data_df['sind'] - med).abs() < TOL * mad]
    return data_df

fig, axes = plt.subplots(len(labels), 1, figsize=(20, 5 * len(labels)))
if len(labels) == 1:
    axes = [axes]

for ax, (fname, lbl) in zip(axes, labels.items()):
    data_df = load_clean(fname)

    ax.plot(data_df['day'], data_df['sind'], '.', color='steelblue',
            markersize=3, alpha=0.7, rasterized=True)

    y_max = data_df['sind'].max()
    y_min = data_df['sind'].min()
    y_pad = (y_max - y_min) * 0.03

    for day in lbl['maxima']:
        ax.axvline(day, color='royalblue', linestyle='--', linewidth=1.2, alpha=0.8)
        ax.text(day, y_max + y_pad, 'MAX', color='royalblue',
                fontsize=7, ha='center', va='bottom', rotation=90)

    for day in lbl['minima']:
        ax.axvline(day, color='darkorange', linestyle='--', linewidth=1.2, alpha=0.8)
        ax.text(day, y_min - y_pad, 'MIN', color='darkorange',
                fontsize=7, ha='center', va='top', rotation=90)

    ax.set_title(fname)
    ax.set_xlabel("Day")
    ax.set_ylabel("S-index")

plt.tight_layout()
plt.show()